# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _Maha Abdulaziz Almolhem_ |
| **Student ID** | _2230002421_ |
| **Section** | _[3]_ |
| **Date submitted** | _10/9/2026_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [2]:
DATA_PATH = "data/spotify_tracks.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (114000, 21)
polars shape: (114000, 21)

columns: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   

         album_name        track_name  popularity  duration_ms  explicit  \
0            Comedy            Comedy          73       230666     False   
1  Ghost (Acoustic)  Ghost - Acoustic          55       149610     False   
2    To Begin Again    To Begin Again          57       210826     False   

   danceability  energy  ...  loudness  mode  speechiness  acousticness  \
0         0.676   

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [3]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [4]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col("track_id").is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col("track_id").is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.051 ms   <- nothing was executed
lazy WITH .collect():      17.212 ms   <- the real cost


**Note:** The CSV has an unnamed index column. Pandas named it `Unnamed: 0`, while Polars kept its name empty, so we used the shared `track_id` column for this comparison.

**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | 0.051 ms |
| Lazy query with `.collect()` | 17.212 ms

In one sentence — why is the first number meaningless?
The first number is meaningless because Polars only created the lazy query plan; it did not read or process the CSV until `.collect()` was called.



---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [5]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "track_genre"    # a text/category column, for grouping
NUMERIC_COL = "danceability"   # a numeric column
NUMERIC_COL2 = "energy"  # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] track_genre
[ok] danceability
[ok] energy


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [6]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:   255.93 ms
polars:    14.18 ms
ratio:     18.05x


 **Observation:** Polars read the CSV in 14.18 ms, while Pandas took 255.93 ms. Polars was approximately 18.05 times faster.

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [7]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 0.58
pandas:     6.85 ms
polars:     1.57 ms
ratio:      4.38x


**Observation:** Filtering tracks with danceability above the median took 6.85 ms in Pandas and 1.57 ms in Polars. Polars was approximately 4.38 times faster.

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [8]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     0.31 ms
polars:     0.77 ms
ratio:      0.40x


**Observation:** Pandas was faster when creating the new column. Pandas took 0.31 ms, while Polars took 0.77 ms, meaning Pandas was approximately 2.48 times faster.

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [9]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:     7.74 ms
polars:     3.02 ms
ratio:      2.56x

pandas result (first rows):
track_genre
acoustic       549.5930
afrobeat       669.5800
alt-rock       534.4930
alternative    559.9270
ambient        367.8668
Name: danceability, dtype: float64

polars result (first rows):
shape: (5, 2)
┌───────────────────┬──────────────┐
│ track_genre       ┆ danceability │
│ ---               ┆ ---          │
│ str               ┆ f64          │
╞═══════════════════╪══════════════╡
│ singer-songwriter ┆ 562.022      │
│ kids              ┆ 778.906      │
│ emo               ┆ 599.321      │
│ comedy            ┆ 577.195      │
│ disney            ┆ 462.8742     │
└───────────────────┴──────────────┘



**Observation:** Polars completed the group-by aggregation in 3.02 ms, while Pandas took 7.74 ms. Polars was approximately 2.56 times faster. Both libraries produced equivalent numerical results, although their row order differed.


**Do both libraries give the same numbers?** Yes. Both libraries produced approximately the same sum for the acoustic genre: 549.5930 in Pandas and 549.5930000000001 in Polars. The tiny difference is caused by floating-point representation. The rows appeared in a different order because Polars does not guarantee the same group order as Pandas.

## 4.5 — Sorting and taking the top N

In [10]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:    26.58 ms
polars:     5.09 ms
ratio:      5.23x


**Observation:** Polars was faster than Pandas when sorting the data and selecting the top 10 rows. Pandas took 26.58 ms, while Polars took 5.09 ms, giving a speedup of 5.23x.

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [11]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):   266.53 ms
polars (lazy + collect):    15.21 ms
ratio:    17.52x


**Look at the query plan.** Polars can show you what it decided to do.

In [12]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("danceability").sum()] BY [col("track_genre")]
  FROM
  simple π 2/2 ["track_genre", "danceability"]
    Csv SCAN [data/spotify_tracks.csv]
    PROJECT 2/21 COLUMNS
    SELECTION: col("danceability") > 0.58
    ESTIMATED ROWS: 141678


**Observation:** In this run, Polars was able to finish the chained pipeline in 15.21 ms, whereas Pandas was able to complete the same pipeline in 266.53 ms, which is about 17.52 times slower. As can be seen in the query plan, only the track_genre and danceability columns are being read by Polars. It also filters the danceability value > 0.58 while scanning the CSV, prior to grouping and aggregation, thus limiting the amount of data passed to subsequent steps.

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [13]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:     6.61 ms
polars:     1.10 ms
ratio:      6.01x

rows before: 114000 -> after: 114000
row count unchanged: True


**Observation:** In this run, the left join was completed in 1.10 ms by Polars compared to 6.61 ms by Pandas, which is about 6.01 times faster than Pandas. The join adds the average danceability for each track’s genre. The Pandas row-count check also identified 114,000 rows as the same count as the previous check.

---
# Part 5 · Results summary

Collect every measurement into one table.

In [14]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,255.93,14.18,18.05,polars
1,filter,6.85,1.57,4.38,polars
2,new column,0.31,0.77,0.40,pandas
3,group by,7.74,3.02,2.56,polars
4,sort + top N,26.58,5.09,5.23,polars
5,chained pipeline,266.53,15.21,17.52,polars
6,join,6.61,1.10,6.01,polars


In [15]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Windows-11-10.0.26200-SP0
   Intel64 Family 6 Model 170 Stepping 4, GenuineIntel


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

The largest difference was in reading the file, where Polars was approximately 18.05 times faster than Pandas: 14.18 ms compared with 255.93 ms. The smallest speedup value was for creating a new column, where Pandas was faster: 0.31 ms compared with 0.77 ms for Polars. A possible explanation is that CSV reading benefits from Polars’ optimized execution, while creating one simple in-memory column is a small operation where additional overhead can have a larger effect.

### Q2 — Did anything surprise you?

Yes. We were surprised that Pandas was faster when creating a new column, taking 0.31 ms compared with 0.77 ms for Polars. We also encountered a ColumnNotFoundError in Part 3 because Pandas named the unnamed CSV column Unnamed: 0, while Polars kept its name empty. We solved the error by using the shared track_id column in the Polars query. This showed us that the two libraries may interpret the same CSV column names differently.

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

We found Pandas easier to write for basic operations because its DataFrame syntax was straightforward. Polars was useful for chaining multiple operations with expressions, but we needed to pay more attention to the Lazy API and when to use .collect(). This helped us in understanding the difference between defining a query and executing it.


### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

We partly reproduced the published claim of a 5x or greater speedup. Polars was 18.05 times faster when reading the file, 5.23 times faster when sorting, 17.52 times faster in the chained pipeline, and 6.01 times faster when joining. However, filtering and grouping were below 5x, and Pandas was faster when creating a new column. Published benchmarks may differ because of dataset size, hardware, library versions, and benchmarking methods.

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.


We would choose a library based on the project requirements, dataset size, performance needs, and compatibility with existing code. Pandas is a reasonable choice for projects that already use its ecosystem or require straightforward data manipulation. For projects involving large CSV files and multiple chained transformations, we would consider Polars because it performed better in most of our measured operations. WE would benchmark both libraries on the actual workload before making a final decision.

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.


DuckDB is an embedded analytical database that allows users to query and analyze tabular data using SQL. It is designed for analytical workloads and supports querying formats such as CSV and Parquet without requiring a separate database server. https://duckdb.org/docs/current/


---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot